# Demonstration 1: the volume-excluded lattice gas

Hard spheres on a lattice, with no reactions and no crowder. This is the exclusion
physics on its own, checked three ways against results that have analytic answers.

| part | measures | moment | against |
|---|---|---|---|
| 1 | excess chemical potential $\mu_{\rm ex}(\eta)$ | first | Carnahan-Starling |
| 2 | occupancy variance $\mathrm{Var}(n)/\langle n\rangle$ | second | Poisson, then the compressibility relation |
| 3 | depletion of a large species by a small one | first, two species | the BMCSL insertion work |

The three parts are not interchangeable. Part 1 fixes the free energy at the mean
density. Part 2 fixes its *curvature*, which part 1 cannot see. Part 3 fixes the
cross-species terms, which a single-species implementation would get wrong while
still passing parts 1 and 2.

**What this validates in the code:** the weighted densities $\xi_0..\xi_3$, the
White-Bear/BMCSL functional, the exact integer table, and the self-exclusion term in
the hop.

Expect a few minutes of runtime for the whole notebook. Each part prints its measured
value beside the prediction, so you check the implementation rather than trust it.

## Symbols, and where they live in the code

Every symbol used below, with the variable that carries it. Energies are in units of
$k_BT$ throughout, so $\beta = 1/k_BT$ never appears explicitly: what the code calls a
free energy is already $\beta F$.

### Lattice and transport

| symbol | code | meaning |
|---|---|---|
| $d$ | `lattice.dim` | dimension, 2 or 3 |
| $h$ | `voxel_nm` | voxel edge length, nm |
| $V$ | `lattice.voxel_volume_nm3` | voxel volume $h^3$, nm$^3$ (cubic even in 2D) |
| $\tau$ | `tau_s` | timestep, s |
| $D$ | `D_um2_s` | diffusion coefficient, $\mu$m$^2$/s |
| $q$ | `sim.hop.q` | baseline per-direction hop probability, $q = D\tau/h^2$ |
| $n_s(v)$ | `state.counts[s, v]` | integer count of species $s$ in voxel $v$ |
| $\rho$ | | number density, here the mean occupancy per voxel |

### Fields and coupling

| symbol | code | meaning |
|---|---|---|
| $\psi_k(v)$ | `psi[k]` | static basis field $k$, supplied by you and never modified |
| $\gamma_{s,k}$ | `Species.gamma` | coupling of species $s$ to field $k$ |
| $\phi_s(v)$ | | the potential species $s$ feels, $\phi_s(v) = \sum_k \gamma_{s,k}\,\psi_k(v)$ |

### Hard spheres

| symbol | code | meaning |
|---|---|---|
| $\sigma_s$ | `sigma_nm` | hard-sphere diameter of species $s$, nm |
| $d\xi^{(k)}_s$ | `exclusion.dxi[s, k]` | per-particle increment, $d\xi^{(k)}_s = \frac{\pi}{6}\sigma_s^k / V$ |
| $\xi_k(v)$ | `exclusion.xi(counts)[k]` | weighted density $k$, for $k = 0,1,2,3$ |
| $\eta$ | | packing fraction, the same thing as $\xi_3$ |
| $\beta F_{\rm ex}$ | `vex.bfex(xi, V)` | White-Bear/BMCSL excess free energy of a voxel |
| $\mu_{\rm ex}$ | `mu_ex_carnahan_starling` | excess chemical potential, $k_BT$ |

### The formulas the code implements

**Weighted densities.** Fundamental-measure theory reduces a hard-sphere mixture to
four scalar fields. On a lattice each is linear in the integer counts:

$$\xi_k(v) = \sum_s n_s(v)\, d\xi^{(k)}_s,
\qquad d\xi^{(k)}_s = \frac{\pi}{6}\frac{\sigma_s^{\,k}}{V},
\qquad k = 0,1,2,3.$$

$\xi_3$ is the packing fraction. $\xi_0$, $\xi_1$ and $\xi_2$ carry number, radius and
surface, which is why a reaction that merges two spheres into one of equal *volume*
still changes the free energy.

**White-Bear / BMCSL excess free energy**, per voxel, in $k_BT$:

$$\beta F_{\rm ex} = V\left[
  -\frac{6}{\pi}\,\xi_0 \ln(1-\xi_3)
  + \frac{18}{\pi}\,\frac{\xi_1 \xi_2}{1-\xi_3}
  + \frac{6}{\pi}\,\frac{\xi_2^3}{\xi_3^2}
    \left(\frac{\xi_3}{(1-\xi_3)^2} + \ln(1-\xi_3)\right)\right].$$

The $\ln(1-\xi_3)$ in the third term is what makes this BMCSL rather than the
Rosenfeld-1989 form. The factor $V$ converts a free-energy density into a per-voxel
energy.

**The hop.** A particle of species $s$ moves from voxel $v$ to a neighbour
$v' = v + e_\delta$ with probability

$$p_\delta(v) = q\; B\!\left(u_\delta(v)\right),
\qquad B(u) = \frac{u}{e^u - 1},$$

where $B$ is the Scharfetter-Gummel (Wang-Peskin-Elston) factor and $u$ is the total
work of the move in $k_BT$:

$$u_\delta(v) = \underbrace{\phi_s(v') - \phi_s(v)}_{\text{field}}
  + \underbrace{\Big[F(\xi_{v'} + d\xi_s) - F(\xi_{v'})\Big]}_{\text{insertion at } v'}
  - \underbrace{\Big[F(\xi_{v}) - F(\xi_{v} - d\xi_s)\Big]}_{\text{removal at } v}.$$

The $-d\xi_s$ in the last bracket is the **self-exclusion**: the hopping particle must
not feel its own volume at the voxel it is leaving. $B(0) = 1$ exactly, so with no work
the hop probability is just $q$.

**Why $\tau$ is small.** $B(u) \to |u|$ as $u \to -\infty$, so a strongly downhill move
can carry a probability far above $q$. The per-direction probabilities plus the stay
probability must sum to at most 1, which gives

$$2\,d\,q\,\mu_{\rm ex}(\eta_{\max}) \le 1$$

rather than the bare $q \le 1/(2d)$. At $\eta = 0.5$, $\mu_{\rm ex} \approx 17\,k_BT$,
so the admissible $\tau$ is an order of magnitude below the CFL bound.
`suggest_tau` computes it.

### Extra symbols for this demonstration

| symbol | code | meaning |
|---|---|---|
| $\mathrm{Var}(n)$ | | variance of single-voxel occupancy, over voxels and time |
| $\langle n \rangle$ | | mean single-voxel occupancy |
| $N$ | | total particle count, fixed |
| $V_{\rm lat}$ | `lattice.n_voxels` | number of voxels, written $V$ in the finite-size factor |

**Part 1 uses** the equilibrium condition that the total chemical potential is uniform:

$$\underbrace{\ln \rho(x)}_{\text{ideal}} + \mu_{\rm ex}(\rho(x)) + \phi(x)
  = \text{const},$$

and the single-species Carnahan-Starling form, which is what BMCSL reduces to:

$$\mu_{\rm ex}(\eta) = \frac{8\eta - 9\eta^2 + 3\eta^3}{(1-\eta)^3}.$$

**Part 2 uses** the compressibility relation, obtained from
$\beta\mu = \ln\rho + \beta\mu_{\rm ex}(\rho)$ and
$\mathrm{Var}(n)/\langle n\rangle = (\rho\,\partial\beta\mu/\partial\rho)^{-1}$:

$$\frac{\mathrm{Var}(n)}{\langle n\rangle}
  = \frac{1}{1 + \eta\,\dfrac{d(\beta\mu_{\rm ex})}{d\eta}}.$$

**Part 3 uses** the same equilibrium condition as part 1, but for the species with
$\gamma = 0$, so the field term drops out:

$$\ln \rho_{\rm big}(x) + \mu_{\rm ex}^{\rm big}(x) = \text{const}.$$

Here $\mu_{\rm ex}^{\rm big}$ is the work of inserting one big sphere into the local
*mixture*, so it depends on both densities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species, mu_ex_carnahan_starling
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import (
    Series, project, mu_ex_from_profile, align_additive_constant,
    relative_discrepancy, report_comparison,
)

VERDICTS = {}          # each part adds its discrepancy here

## Part 1: the excess chemical potential

At equilibrium the total chemical potential is uniform in space:

$$\ln \rho(x) + \mu_{\rm ex}(\rho(x)) + \phi(x) = \text{const}.$$

So a density profile in a known potential gives $\mu_{\rm ex}$ directly. For one
species BMCSL reduces to Carnahan-Starling,

$$\mu_{\rm ex}(\eta) = \frac{8\eta - 9\eta^2 + 3\eta^3}{(1-\eta)^3},$$

and only one free parameter, the additive constant, is fitted. The shape is
parameter-free.

**This is also the self-exclusion test.** A hopping particle must not feel its own
volume at the voxel it leaves. Dropping that term changes the *shape* of the recovered
curve, so it cannot hide in the fitted constant.

The timestep is not set by the CFL bound here. With exclusion the constraint is the
downhill Bernoulli factor, $2\,d\,q\,\mu_{\rm ex}(\eta_{\max}) \le 1$, which at
$\eta = 0.5$ is an order of magnitude tighter. `suggest_tau` computes it.

In [ ]:
EOS_SHAPE     = (32, 24)
EOS_VOXEL_NM  = 20.0
EOS_SIGMA_NM  = 8.0
EOS_CAP       = 20
EOS_GAMMA     = 3.0
EOS_PER_VOXEL = 3           # seeded exactly, so there are no high-occupancy outliers
EOS_STEPS     = 60_000
EOS_SAMPLE_EVERY = 25

eos_dxi3 = (np.pi / 6) * EOS_SIGMA_NM ** 3 / EOS_VOXEL_NM ** 3
EOS_TAU = suggest_tau(D_um2_s=1.0, voxel_nm=EOS_VOXEL_NM, dim=2,
                      eta_max=3.0 * EOS_PER_VOXEL * eos_dxi3)

print(f"one particle contributes dxi3 = {eos_dxi3:.5f}")
print(f"mean packing fraction        = {EOS_PER_VOXEL * eos_dxi3:.4f}")
print(f"suggested tau                = {EOS_TAU:.3e} s")

eos_ramp = np.arange(EOS_SHAPE[-1], dtype=float) / EOS_SHAPE[-1]
eos_psi  = np.broadcast_to(eos_ramp, EOS_SHAPE).copy()[None, ...]

eos_sim = Simulation(
    shape=EOS_SHAPE, voxel_nm=EOS_VOXEL_NM,
    species=[Species("A", sigma_nm=EOS_SIGMA_NM, gamma=np.array([EOS_GAMMA]))],
    occupancy_cap=EOS_CAP, psi=eos_psi, D_um2_s=1.0, tau_s=EOS_TAU, seed=0,
)
eos_sim.set_counts("A", np.full(EOS_SHAPE, EOS_PER_VOXEL, dtype=np.int64))
eos_sim.record_initial()

eos_rho = Series("density")
for i in range(EOS_STEPS):
    eos_sim.step()
    if i >= EOS_STEPS // 3 and i % EOS_SAMPLE_EVERY == 0:
        eos_rho.add(project(eos_sim.state.lattice_view("A"), eos_sim.lattice)
                    / EOS_SHAPE[0])
eos_sim.state.check_mass()
print(f"{eos_rho.n} samples; density {eos_rho.mean.min():.2f} to "
      f"{eos_rho.mean.max():.2f} per voxel")

In [ ]:
eos_density  = eos_rho.mean
eos_eta      = eos_density * eos_dxi3
eos_analytic = mu_ex_carnahan_starling(eos_eta)
eos_measured = align_additive_constant(
    mu_ex_from_profile(eos_density, EOS_GAMMA * eos_ramp), eos_analytic)
eos_sem = eos_rho.sem / eos_density        # d(mu_ex)/d(rho) = -1/rho

print(f"packing fraction spans {eos_eta.min():.3f} to {eos_eta.max():.3f}")
print(f"mu_ex spans            {eos_analytic.min():.3f} to "
      f"{eos_analytic.max():.3f} kT")
print()
print(report_comparison("Part 1: mu_ex(eta) vs Carnahan-Starling",
                        eos_measured, eos_analytic, sem=eos_sem))

VERDICTS["part 1: mu_ex vs Carnahan-Starling"] = relative_discrepancy(
    eos_measured, eos_analytic)["max"]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
viz.plot_mu_ex(eos_eta, eos_measured, eos_analytic, sem=eos_sem, ax=ax[0])
ax[0].set_title("Part 1: excess chemical potential", fontsize=10)
viz.plot_profile(eos_density, sem=eos_rho.sem, ax=ax[1],
                 xlabel="voxel along the field axis",
                 ylabel="mean occupancy per voxel",
                 label_measured="measured density",
                 title="the profile it came from")
plt.tight_layout(); plt.show()

## Part 2: the occupancy distribution is sub-Poissonian

Part 1 pinned the free energy at the mean density. It says nothing about the
fluctuations, and a functional with the right value but the wrong curvature would pass
it. The variance is the observable that sees the curvature.

An ideal lattice gas places each particle independently, so single-voxel occupancy is
Poisson and $\mathrm{Var}(n)/\langle n\rangle = 1$. Hard spheres cannot: a voxel that
already holds particles is expensive to add to, so the distribution narrows. The
compressibility relation says by how much:

$$\frac{\mathrm{Var}(n)}{\langle n\rangle}
  = \frac{1}{1 + \eta\,\dfrac{d(\beta\mu_{\rm ex})}{d\eta}}.$$

Ideal gas: $\mu_{\rm ex} = 0$, ratio 1. Hard spheres: $\mu_{\rm ex}' > 0$, ratio below
1. Carnahan-Starling supplies the derivative, so the prediction is parameter-free.

**Two finite-size corrections, both stated rather than hidden.** The lattice is
canonical, so the variance of the *total* is zero; the relation applies to one voxel
because the rest of the lattice acts as its reservoir, with a correction $(1 - 1/V)$.
And the ideal control is multinomial, so its exact ratio is also $(1 - 1/V)$, not 1.
At $V = 4096$ both are $0.02\%$, and the predicted ideal value uses the corrected
figure.

In [ ]:
FLUCT_SHAPE    = (64, 64)
FLUCT_VOXEL_NM = 20.0
FLUCT_SIGMA_NM = 8.0
FLUCT_CAP      = 20
FLUCT_OCCUPANCIES = (3, 6)          # eta = 0.10 and 0.20
FLUCT_STEPS    = 40_000
FLUCT_SAMPLE_EVERY = 100            # occupancy decorrelates slowly

FLUCT_V     = FLUCT_SHAPE[0] * FLUCT_SHAPE[1]
fluct_dxi3  = (np.pi / 6) * FLUCT_SIGMA_NM ** 3 / FLUCT_VOXEL_NM ** 3
FINITE_SIZE = 1.0 - 1.0 / FLUCT_V

def predicted_ratio(eta, h=1e-6):
    if eta == 0.0:
        return FINITE_SIZE
    dmu = (mu_ex_carnahan_starling(eta + h)
           - mu_ex_carnahan_starling(eta - h)) / (2 * h)
    return FINITE_SIZE / (1.0 + eta * dmu)

print(f"finite-size factor (1 - 1/V) = {FINITE_SIZE:.5f}")
for n in FLUCT_OCCUPANCIES:
    print(f"  {n} per voxel -> eta = {n * fluct_dxi3:.3f}, "
          f"predicted Var/mean = {predicted_ratio(n * fluct_dxi3):.4f}")

In [ ]:
def measure_fluctuations(n_per_voxel, excluded):
    """Mean and variance of single-voxel occupancy at equilibrium.

    No field, so every voxel is equivalent and the occupancy can be pooled over all
    of them at each sample. That is what makes a variance measurable quickly.
    """
    eta = n_per_voxel * fluct_dxi3 if excluded else 0.0
    tau = (suggest_tau(1.0, FLUCT_VOXEL_NM, 2, max(3.0 * eta, 0.05))
           if excluded else 2e-5)
    sim = Simulation(
        shape=FLUCT_SHAPE, voxel_nm=FLUCT_VOXEL_NM,
        species=[Species("A", FLUCT_SIGMA_NM if excluded else 0.0, np.zeros(1))],
        occupancy_cap=FLUCT_CAP if excluded else 10_000,
        psi=np.zeros((1,) + FLUCT_SHAPE), D_um2_s=1.0, tau_s=tau,
        exclusion=excluded, seed=1, attach_log_handler=False,
    )
    sim.set_counts("A", np.full(FLUCT_SHAPE, n_per_voxel, dtype=np.int64))
    sim.record_initial()
    s = s2 = 0.0
    n = 0
    for i in range(FLUCT_STEPS):
        sim.step()
        if i >= FLUCT_STEPS // 3 and i % FLUCT_SAMPLE_EVERY == 0:
            c = sim.state.counts[0].astype(np.float64)
            s += c.sum(); s2 += (c * c).sum(); n += c.size
    sim.state.check_mass()
    mean = s / n
    return dict(n_pv=n_per_voxel, excluded=excluded, eta=eta,
                mean=mean, var=s2 / n - mean * mean,
                ratio=(s2 / n - mean * mean) / mean)

fluct = [measure_fluctuations(n, e)
         for n in FLUCT_OCCUPANCIES for e in (False, True)]
print(f"{len(fluct)} runs complete")

In [ ]:
print(f"{'case':>10} {'n/voxel':>8} {'eta':>7} {'mean':>7} {'var':>7} "
      f"{'Var/mean':>9} {'predicted':>10} {'error':>7}")
for r in fluct:
    pred = predicted_ratio(r["eta"])
    print(f"{'excluded' if r['excluded'] else 'ideal':>10} {r['n_pv']:>8} "
          f"{r['eta']:>7.3f} {r['mean']:>7.3f} {r['var']:>7.3f} "
          f"{r['ratio']:>9.4f} {pred:>10.4f} "
          f"{abs(r['ratio']/pred - 1)*100:>6.1f}%")

ideal = [r for r in fluct if not r["excluded"]]
excl  = [r for r in fluct if r["excluded"]]
print()
print(report_comparison("Part 2, ideal gas: Var/mean vs Poisson",
                        float(np.mean([r["ratio"] for r in ideal])), FINITE_SIZE))
for r in excl:
    print()
    print(report_comparison(
        f"Part 2, excluded at eta = {r['eta']:.3f}: Var/mean vs compressibility",
        r["ratio"], predicted_ratio(r["eta"])))

VERDICTS["part 2: ideal is Poisson"] = max(
    abs(r["ratio"] / FINITE_SIZE - 1) for r in ideal)
VERDICTS["part 2: excluded matches compressibility"] = max(
    abs(r["ratio"] / predicted_ratio(r["eta"]) - 1) for r in excl)

In [ ]:
eta_curve = np.linspace(0.0, 0.30, 200)
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8))

ax[0].plot(eta_curve, [predicted_ratio(e) for e in eta_curve], "-", lw=1.8,
           color="#c1440e", label="compressibility relation", zorder=2)
ax[0].axhline(FINITE_SIZE, ls="--", lw=1.2, color="#888", label="Poisson (ideal)")
ax[0].plot([r["eta"] for r in excl], [r["ratio"] for r in excl], "o", ms=7,
           color="#1f4e79", label="measured, exclusion on", zorder=3)
ax[0].plot([0.0] * len(ideal), [r["ratio"] for r in ideal], "s", ms=7,
           color="#2e7d32", label="measured, ideal", zorder=3)
ax[0].set_xlabel(r"packing fraction $\eta = \xi_3$")
ax[0].set_ylabel(r"$\mathrm{Var}(n)\,/\,\langle n\rangle$")
ax[0].set_title("Part 2: exclusion narrows the distribution", fontsize=10)
ax[0].set_ylim(0, 1.15)
ax[0].legend(frameon=False, fontsize=8.5)

n_pv = FLUCT_OCCUPANCIES[-1]
for excluded, colour, label in ((False, "#2e7d32", "ideal"),
                                (True, "#1f4e79", "exclusion on")):
    eta = n_pv * fluct_dxi3 if excluded else 0.0
    tau = (suggest_tau(1.0, FLUCT_VOXEL_NM, 2, max(3.0 * eta, 0.05))
           if excluded else 2e-5)
    sim = Simulation(
        shape=FLUCT_SHAPE, voxel_nm=FLUCT_VOXEL_NM,
        species=[Species("A", FLUCT_SIGMA_NM if excluded else 0.0, np.zeros(1))],
        occupancy_cap=FLUCT_CAP if excluded else 10_000,
        psi=np.zeros((1,) + FLUCT_SHAPE), D_um2_s=1.0, tau_s=tau,
        exclusion=excluded, seed=2, attach_log_handler=False,
    )
    sim.set_counts("A", np.full(FLUCT_SHAPE, n_pv, dtype=np.int64))
    sim.record_initial()
    for i in range(FLUCT_STEPS // 2):
        sim.step()
    c = sim.state.counts[0]
    ax[1].hist(c, bins=np.arange(-0.5, c.max() + 1.5), density=True,
               histtype="step", lw=1.8, color=colour,
               label=f"{label}  (var/mean = {c.var()/c.mean():.2f})")
ax[1].set_xlabel("occupancy of a single voxel")
ax[1].set_ylabel("probability")
ax[1].set_title(f"Occupancy at {n_pv} particles per voxel", fontsize=10)
ax[1].legend(frameon=False, fontsize=8.5)
plt.tight_layout(); plt.show()

## Part 3: a mixture, and depletion

Parts 1 and 2 use one species. Both would pass with the cross-species terms of the
functional wrong. This part uses two hard-sphere species of different size, and
couples **only the smaller one** to the field. The large species feels no field at
all, so any structure in its profile can only come from volume exclusion.

Its equilibrium condition is therefore

$$\ln \rho_{\rm big}(x) + \mu_{\rm ex}^{\rm big}(x) = \text{const},$$

where $\mu_{\rm ex}^{\rm big}$ is the work of inserting a big sphere into the *local
mixture*. That depends on both densities, so it is a genuinely multi-species
quantity. This is depletion: the big spheres are pushed out of the region the small
ones crowd.

The occupancy cap is chosen from the **largest** species. Twenty 8 nm spheres in a
20 nm voxel give $\eta = 0.67$, which puts exclusion, not the cap, in charge.

In [ ]:
MIX_SHAPE     = (32, 24)
MIX_VOXEL_NM  = 20.0
MIX_SIGMA_SMALL, MIX_SIGMA_BIG = 5.0, 8.0      # volume ratio about 4.1
MIX_CAP       = 20
MIX_GAMMA_SMALL = 1.5
MIX_SMALL_PER_VOXEL, MIX_BIG_PER_VOXEL = 4, 2
MIX_STEPS     = 80_000
MIX_SAMPLE_EVERY = 25

MIX_TAU = suggest_tau(D_um2_s=1.0, voxel_nm=MIX_VOXEL_NM, dim=2, eta_max=0.45)
print(f"tau = {MIX_TAU:.3e} s")

mix_ramp = np.arange(MIX_SHAPE[-1], dtype=float) / MIX_SHAPE[-1]
mix_psi  = np.broadcast_to(mix_ramp, MIX_SHAPE).copy()[None, ...]

mix_sim = Simulation(
    shape=MIX_SHAPE, voxel_nm=MIX_VOXEL_NM,
    species=[
        Species("small", sigma_nm=MIX_SIGMA_SMALL,
                gamma=np.array([MIX_GAMMA_SMALL])),
        Species("big",   sigma_nm=MIX_SIGMA_BIG, gamma=np.array([0.0])),
    ],
    occupancy_cap=MIX_CAP, psi=mix_psi, D_um2_s=1.0, tau_s=MIX_TAU, seed=0,
)
mix_sim.set_counts("small", np.full(MIX_SHAPE, MIX_SMALL_PER_VOXEL, dtype=np.int64))
mix_sim.set_counts("big",   np.full(MIX_SHAPE, MIX_BIG_PER_VOXEL,   dtype=np.int64))
mix_sim.record_initial()

mix_small, mix_big = Series("small"), Series("big")
for i in range(MIX_STEPS):
    mix_sim.step()
    if i >= MIX_STEPS // 3 and i % MIX_SAMPLE_EVERY == 0:
        mix_small.add(project(mix_sim.state.lattice_view("small"),
                              mix_sim.lattice) / MIX_SHAPE[0])
        mix_big.add(project(mix_sim.state.lattice_view("big"),
                            mix_sim.lattice) / MIX_SHAPE[0])
mix_sim.state.check_mass()

rho_s, rho_b = mix_small.mean, mix_big.mean
print(f"{mix_small.n} samples")
print(f"small  {rho_s.min():.2f} to {rho_s.max():.2f} per voxel   (field-coupled)")
print(f"big    {rho_b.min():.2f} to {rho_b.max():.2f} per voxel   (gamma = 0)")
print(f"the big species varies by {(rho_b.max()/rho_b.min() - 1)*100:.1f}% "
      f"across the box, from exclusion alone")

In [ ]:
# The composition is a time-averaged mean, so it is fractional. That takes the
# elementwise free-energy path rather than the integer table, which is correct: a
# fractional count vector cannot index the table.
mix_comp = np.vstack([rho_s, rho_b])
mix_exc  = mix_sim.exclusion
dnu_big  = np.array([0, 1], dtype=np.int64)
mu_big = (mix_exc.shifted_free_energy(mix_comp,
                                      mix_exc.stoichiometry_offset(dnu_big), dnu_big)
          - mix_exc.free_energy(mix_comp))

mix_measured = align_additive_constant(
    mu_ex_from_profile(rho_b, np.zeros_like(rho_b)), mu_big)
mix_sem = mix_big.sem / rho_b

print(f"insertion work for a big sphere: {mu_big.min():.3f} to {mu_big.max():.3f} kT")
print()
print(report_comparison(
    "Part 3: -ln rho_big vs the BMCSL big-sphere insertion work",
    mix_measured, mu_big, sem=mix_sem))

VERDICTS["part 3: big-sphere insertion work"] = relative_discrepancy(
    mix_measured, mu_big)["max"]

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.5))
viz.plot_profile(rho_s, sem=mix_small.sem, ax=ax[0], ylabel="mean occupancy",
                 label_measured="small (field-coupled)",
                 title="small species: driven by the field")
viz.plot_profile(rho_b, sem=mix_big.sem, ax=ax[1], ylabel="mean occupancy",
                 label_measured=r"big ($\gamma=0$)",
                 title="big species: driven by depletion only")
viz.plot_profile(mix_measured, predicted=mu_big, sem=mix_sem, ax=ax[2],
                 ylabel=r"$\mu_{\rm ex}^{\rm big}$  ($k_BT$)",
                 label_measured=r"$-\ln\rho_{\rm big}$ + const",
                 label_predicted="BMCSL insertion work",
                 title="and they agree")
plt.tight_layout(); plt.show()

## Summary

In [ ]:
print("Demonstration 1: the volume-excluded lattice gas\n")
for label, value in VERDICTS.items():
    print(f"  {value*100:6.2f}%   {label}")
print(f"\n  {len(VERDICTS)} checks, worst discrepancy {max(VERDICTS.values())*100:.2f}%")

## What to take from this

Three independent properties of one free energy, all matching analytic results with
at most one fitted constant.

The parts are ordered so that each closes a gap the previous one leaves open. Part 1
fixes $\mu_{\rm ex}$ at the mean density. Part 2 fixes its derivative, because the
variance depends on $d\mu_{\rm ex}/d\eta$ and part 1 is blind to it. Part 3 fixes the
cross-species terms, which neither of the single-species parts constrains at all.
Passing all three means the functional is right, not merely tangent to right.

**Try changing:**

- `EOS_GAMMA = 0`: the profile goes flat, the packing-fraction range collapses, and
  part 1 has nothing left to compare. `relative_discrepancy` refuses a flat
  prediction rather than returning a meaningless number.
- `FLUCT_SIGMA_NM = 4` at the same occupancy: smaller spheres give a smaller $\eta$,
  so the distribution moves back toward Poisson.
- `exclusion=False` in `measure_fluctuations` while leaving `FLUCT_SIGMA_NM = 8`: the
  diameters are then ignored and the ratio returns to 1. That confirms the narrowing
  comes from the free energy and not from the occupancy cap.
- `MIX_SIGMA_BIG = MIX_SIGMA_SMALL`: no size asymmetry, so no depletion. The big
  profile should flatten.
- `MIX_SIGMA_BIG = 12`: twenty 12 nm spheres in a 20 nm voxel exceed a packing
  fraction of 1. Construction refuses it and names the largest admissible cap.
- `EOS_SIGMA_NM = 20` with `EOS_VOXEL_NM = 20`: a sphere as wide as the voxel admits
  one particle per voxel, so no crowding study is possible. The guard says so.